In [1]:
import glob
import io
import os
import shutil
import sys
import time
import pathlib
import re
from collections import OrderedDict
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import polars as pl
import pyautogui
import win32clipboard

from PIL import Image, ImageGrab

from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoAlertPresentException,
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException
)

from IPython.display import HTML


In [2]:
def input_data(data_dir):
    list_files = []
    for path in pathlib.Path(data_dir).glob('**/*.*'):
        if path.suffix.lower() in ['.xlsx', '.csv'] and path.stat().st_size > 0:
            export_time = datetime.fromtimestamp(path.stat().st_mtime)
            try:
                df = pl.read_excel(path) if path.suffix.lower() == '.xlsx' else pl.read_csv(path, infer_schema_length=10000, ignore_errors=True)
                if not df.is_empty():
                    df = df.with_columns([
                        pl.lit(path.stem).alias('sheet_name'),
                        pl.lit(export_time).alias('Export time')
                    ])
                    list_files.append(df)
            except:
                continue
                
    return pl.concat(list_files, how="diagonal_relaxed") if list_files else pl.DataFrame()

In [3]:
first_glob = os.path.expanduser("~").replace("//", "/")

folder_paths = {
    "req": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OU.xlsx",
    "current_agents": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_agent/",
    "input_iex":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_iex/',
}

hc_extend_combination = (
    pl.read_parquet(
       f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/hc_extend_combination.parquet'
    )
    .select([
        "Date",
        "IEX ID",
        "Employee Name",
        "LOB",
        "Email Id",
        "Supervisor Name"
    ])
    .with_columns([
        pl.col("Date").cast(pl.Date, strict=False),

        pl.col("IEX ID")
        .str.replace_all('"', "")
        .cast(pl.Int64, strict=False)
    ])
    .filter(
        pl.col("Date") >= pl.date(2026, 1, 1)
    )
)

hc_extend_combination

Date,IEX ID,Employee Name,LOB,Email Id,Supervisor Name
date,i64,str,str,str,str
2026-06-01,3035065,"""TRAN THI TUYET NHUNG""","""Non_Lodging_Training""","""thituyetnhung.tran@concentrix.…","""Tran Tran"""
2026-06-01,3002199,"""LE HOANG MAI THY""","""Lodging""","""hoangmaithy.le@concentrix.com""","""Tran Thao Uyen"""
2026-06-01,3092114,"""PHAN DAT LOI""","""Lodging""","""datloi.phan1@concentrix.com""","""Chau Thien Kim"""
2026-06-01,3029716,"""DANG NU NGOC TRINH""","""Lodging_Nesting""","""nungoctrinh.dang@concentrix.co…","""Tran Tran"""
2026-06-01,3002196,"""PHUNG MY UYEN""","""Lodging""","""myuyen.phung@concentrix.com""","""Truong Thien Thanh Toan"""
…,…,…,…,…,…
2026-01-31,null,"""PHAM DUONG HOANG VU""","""Support""","""duonghoangvu.pham@concentrix.c…","""Tran Van"""
2026-01-31,3092114,"""PHAN DAT LOI""","""Lodging""","""datloi.phan1@concentrix.com""","""Chau Thien Kim"""
2026-01-31,3085252,"""HOANG LE VY""","""Lodging""","""levy.hoang@concentrix.com""","""Ann"""


In [4]:
def process_ou_hours(file_path):
    try:
        df_raw = pl.read_excel(source=file_path, has_header=False, infer_schema_length=0)
        
        header_vals = df_raw.row(0)
        new_columns = [
            val.strftime("%Y-%m-%d") if isinstance(val, datetime) else str(val).strip() if val is not None else "Unknown" 
            for val in header_vals
        ]
        df_raw.columns = new_columns
        
        df = df_raw.slice(1).with_columns(
            pl.col("Interval").cast(pl.String).str.replace(r"^.*1899-12-31\s+", "").str.slice(0, 8).alias("Interval")
        )
        
        final_df = (df.unpivot(index=["LOB", "Site", "Interval"], variable_name="Date_Str", value_name="Value")
            .with_columns([
                (pl.col("Date_Str").str.slice(0, 10) + " " + pl.col("Interval"))
                .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                .dt.truncate("1m").alias("PST_Datetime"),
                
                pl.col("Value").cast(pl.Float64, strict=False).fill_null(0.0).alias("OU Req by Site"),
                pl.col("LOB").str.strip_chars(), 
                pl.col("Site").str.strip_chars()
            ])
            .with_columns(
                pl.col("PST_Datetime")
                .dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest")
                .alias("_tz_aware")
            )
            .with_columns([
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).alias("VNT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).alias("CLT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).alias("IST_Datetime"),
                pl.col("OU Req by Site").sum().over(["LOB", "PST_Datetime"]).alias("Total OU Req")
            ])
            .drop("_tz_aware")
        )
        return final_df.select(["LOB", "Site", "PST_Datetime", "VNT_Datetime", "CLT_Datetime", "IST_Datetime", "Total OU Req", "OU Req by Site"])
    except Exception as e:
        print(f"Error processing file: {e}")
        return pl.DataFrame()

req_path = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\OU.xlsx")
df_req = process_ou_hours(folder_paths["req"])

In [5]:
REQ_SITE_MAPPING = {
    "Concentrix (Pune)": "PUN",
    "Concentrix (Ho Chi Minh City)": "HCM",
    "Concentrix (Kolkata)": "KOL",
    "Concentrix (Cairo)": "CAI"
}

SITE_MAPPING = {"Pune": "PUN", "Ho Chi Minh": "HCM", "Kolkata": "KOL", "Cairo": "CAI"}

LOB_MAPPING = {
    "Lodging Chat": ["Chat_OD_EN_Car_Activity", "Chat_OD_EN_Lodging", "Chat - Global English Lodging Nesting", "Chat_Lodging English w Car"],
    "Non Lodging Chat": ["Chat - Global English Non- Lodging Nesting", "Chat_OD_EN_Dual_GDS", "Chat_OD_EN_Amadeus"]
}

ACTIVE_STATES = ["AVAILABLECHAT", "OUTBOUNDCALL"]

outage_db = (
    input_data(folder_paths["current_agents"])
    .filter(
        pl.col("Export time").is_in(
            input_data(folder_paths["current_agents"])
            .select("Export time").unique().sort("Export time", descending=True).limit(16).get_column("Export time")
        )
    )
    .select([
        "Business Location", "Export time", "Agent Name", "Agent Email", "Agent Manager",
        "Connect State", "Assigned Workitem Count", "Duration", "Queue Group / Routing Profile"
    ])
    .with_columns([
        pl.col("Export time").cast(pl.Datetime, strict=False).dt.truncate("30m").alias("VNT_Datetime"),
        pl.col("Duration").str.strptime(pl.Time, "%H:%M:%S", strict=False)
    ])
    .filter(pl.col("Export time") == pl.col("Export time").max().over("VNT_Datetime"))
    .with_columns([
        pl.col("VNT_Datetime").dt.replace_time_zone("Asia/Ho_Chi_Minh").dt.convert_time_zone("America/Los_Angeles").dt.replace_time_zone(None).alias("PST_Datetime")
    ])
    .with_columns([
        pl.when(pl.col("Queue Group / Routing Profile").is_in(LOB_MAPPING["Lodging Chat"])).then(pl.lit("Lodging Chat"))
        .when(pl.col("Queue Group / Routing Profile").is_in(LOB_MAPPING["Non Lodging Chat"])).then(pl.lit("Non Lodging Chat"))
        .otherwise(None).alias("LOB")
    ])
    .filter(pl.col("LOB").is_not_null())
)

active_cond = pl.col("Connect State").is_in(ACTIVE_STATES) | (pl.col("Assigned Workitem Count").cast(pl.Int32, strict=False).fill_null(0) > 0)

inactive_cond = ~active_cond

def add_time_cols(df, col="PST_Datetime"):
    return (
        df.with_columns(pl.col(col).dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest").alias("_tz"))
        .with_columns([
            pl.col(col).dt.date().alias("PST_Date"),
            pl.col(col).dt.strftime("%H:%M").alias("PST"),
            pl.col("_tz").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("IST"),
            pl.col("_tz").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("CLT"),
            pl.col("_tz").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("VNT")
        ])
        .drop("_tz")
    )

def build_hc(df, cond, label):
    value_col = f"Heads {label}"
    return (
        df.group_by(["LOB", "PST_Datetime"])
        .agg([
            pl.col("Agent Name").filter(pl.col("Business Location").str.contains(loc) & cond).n_unique().alias(prefix)
            for loc, prefix in SITE_MAPPING.items()
        ])
        .fill_null(0)
        .sort(["LOB", "PST_Datetime"])
        .unpivot(index=["LOB", "PST_Datetime"], on=list(SITE_MAPPING.values()), variable_name="Site", value_name=value_col)
        .pipe(add_time_cols)
        .select(["LOB", "Site", value_col, "PST_Datetime", "PST_Date", "PST", "IST", "CLT", "VNT"])
        .sort(["LOB", "Site"])
    )

df_active = build_hc(outage_db, active_cond, "Active")
df_inactive = build_hc(outage_db, inactive_cond, "Inactive")

df_req = (
    df_req
    .with_columns([
        pl.col("Site").replace(REQ_SITE_MAPPING),
        
        pl.when(pl.col("LOB") == "Lodging chat").then(pl.lit("Lodging Chat"))
        .when(pl.col("LOB") == "Non-Lodging chat").then(pl.lit("Non Lodging Chat"))
        .otherwise(pl.col("LOB"))
        .alias("LOB"),
        
        (pl.col("Total OU Req") * 2).alias("Total OU Req"),
        (pl.col("OU Req by Site") * 2).alias("OU Req by Site")
    ])
    .select(["LOB", "Site", "PST_Datetime", "Total OU Req", "OU Req by Site"])
)

df_active = (
    df_active
    .join(df_req, on=["LOB", "Site", "PST_Datetime"], how="left")
    .with_columns([
        (pl.col("Heads Active") - pl.col("OU Req by Site")).round(1).alias("Missing Heads"),
        pl.when(
            (pl.col("OU Req by Site").is_null()) |
            (pl.col("OU Req by Site") == 0)
        )
        .then(pl.lit("-"))
        .otherwise(
            (((pl.col("Heads Active") / pl.col("OU Req by Site")) * 100).round(1).cast(pl.String) + "%")
        )
        .alias("Heads Active / OU Req by Site"),
        pl.col("Total OU Req").round(1),
        pl.col("OU Req by Site").round(1)
    ])
    .select([
        "LOB", "Site", "Total OU Req", "OU Req by Site", "Heads Active",
        "Missing Heads", "Heads Active / OU Req by Site",
        "PST_Datetime", "PST_Date", "PST", "IST", "CLT", "VNT"
    ])
)

df_inactive_details = (
    outage_db
    .filter(inactive_cond)
    .select([
        "Agent Name", "Agent Email", "LOB", "Business Location",
        "Connect State", "Duration", "PST_Datetime"
    ])
    .pipe(add_time_cols)
)

latest_dt = outage_db.select(pl.col("PST_Datetime").max()).item()

df_active = df_active.filter(pl.col("PST_Datetime") == latest_dt)
df_inactive = df_inactive.filter(pl.col("PST_Datetime") == latest_dt)
df_inactive_details = df_inactive_details.filter(pl.col("PST_Datetime") == latest_dt)

df_lodging = df_inactive_details.filter(pl.col("LOB") == "Lodging Chat").sort(["Business Location", "Agent Name"])
df_non_lodging = df_inactive_details.filter(pl.col("LOB") == "Non Lodging Chat").sort(["Business Location", "Agent Name"])

with pl.Config(tbl_rows=-1):
    display(df_inactive_details)
    display(df_lodging)
    display(df_non_lodging)
    display(df_active)
    display(df_inactive)

Agent Name,Agent Email,LOB,Business Location,Connect State,Duration,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,str,str,str,time,datetime[μs],date,str,str,str,str
"""Bui, Ba Duong""","""baduong.bui@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:04:27,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Chakraborty, Swapnee""","""swapneel.chakraborty@concentri…","""Lodging Chat""","""Concentrix (Kolkata)""","""ENDOFSHIFT""",02:24:46,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Chau, Thien Kim""","""thienkim.chau@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""ENDOFSHIFT""",03:40:22,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""HALDER, ANTHONY""","""anthony.halder@concentrix.com""","""Lodging Chat""","""Concentrix (Kolkata)""","""ENDOFSHIFT""",00:00:54,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hieu, Nguyen""","""tronghieu.nguyen7@concentrix.c…","""Non Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",00:00:11,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hoang, Le Vy""","""levy.hoang@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:47:36,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""KHAN, SHAGUFTA""","""shagufta.khan2@concentrix.com""","""Lodging Chat""","""Concentrix (Pune)""","""LOGIN""",12:32:07,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, Hoang Mai Thy""","""hoangmaithy.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:52:23,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, MINH QUAN""","""minhquan.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",00:01:37,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""


Agent Name,Agent Email,LOB,Business Location,Connect State,Duration,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,str,str,str,time,datetime[μs],date,str,str,str,str
"""Bui, Ba Duong""","""baduong.bui@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",05:04:27,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Chau, Thien Kim""","""thienkim.chau@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""ENDOFSHIFT""",03:40:22,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hoang, Le Vy""","""levy.hoang@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:47:36,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, Hoang Mai Thy""","""hoangmaithy.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:52:23,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, MINH QUAN""","""minhquan.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",00:01:37,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Le, Thuy Trieu""","""thuytrieu.le@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:06:51,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Ngo, Ha Trang""","""hatrang.ngo@concentrix.com""","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:41:29,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Nguyen, Thi Thien An""","""thithienan.nguyen@concentrix.c…","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""TRAINING""",04:32:21,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Nguyen, Tran Anh Thu""","""trananhthu.nguyen2@concentrix.…","""Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LUNCH""",00:32:09,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""


Agent Name,Agent Email,LOB,Business Location,Connect State,Duration,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,str,str,str,time,datetime[μs],date,str,str,str,str
"""Refaat Ahmed, Youssef""","""youssef.refaatahmed@concentrix…","""Non Lodging Chat""","""Concentrix (Cairo)""","""ENDOFSHIFT""",09:26:18,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Sayed Abdelhadi, Injy Ehab""","""injy.sayedabdelhadi@concentrix…","""Non Lodging Chat""","""Concentrix (Cairo)""","""ENDOFSHIFT""",04:07:24,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Hieu, Nguyen""","""tronghieu.nguyen7@concentrix.c…","""Non Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""LOGIN""",00:00:11,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Phan, Thi Kim Tuyen""","""thikimtuyen.phan@concentrix.co…","""Non Lodging Chat""","""Concentrix (Ho Chi Minh City)""","""BREAK""",00:12:31,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Mukherjee, Swarnali""","""swarnali.mukherjee@concentrix.…","""Non Lodging Chat""","""Concentrix (Kolkata)""","""LOGIN""",11:00:13,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""


LOB,Site,Total OU Req,OU Req by Site,Heads Active,Missing Heads,Heads Active / OU Req by Site,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,f64,f64,u32,f64,str,datetime[μs],date,str,str,str,str
"""Lodging Chat""","""CAI""",null,null,0,null,"""-""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""HCM""",20.8,20.8,8,-12.8,"""38.4%""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""KOL""",20.8,0.0,2,2.0,"""-""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""PUN""",20.8,0.0,1,1.0,"""-""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""CAI""",19.5,9.9,0,-9.9,"""0.0%""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""HCM""",19.5,3.3,8,4.7,"""240.2%""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""KOL""",19.5,6.3,1,-5.3,"""15.9%""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""PUN""",19.5,0.0,0,0.0,"""-""",2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""


LOB,Site,Heads Inactive,PST_Datetime,PST_Date,PST,IST,CLT,VNT
str,str,u32,datetime[μs],date,str,str,str,str
"""Lodging Chat""","""CAI""",0,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""HCM""",12,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""KOL""",5,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Lodging Chat""","""PUN""",2,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""CAI""",2,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""HCM""",2,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""KOL""",1,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""
"""Non Lodging Chat""","""PUN""",0,2026-05-11 21:00:00,2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00"""


In [6]:
raw = input_data(folder_paths["input_iex"])

IEX = (
    raw.select([c for c in raw.columns if c != "__UNNAMED__4"])
    .rename({
        "Agent Schedules": "Agent", "__UNNAMED__1": "Date", "__UNNAMED__2": "Start_Shift",
        "__UNNAMED__3": "End_Shift", "__UNNAMED__5": "Scheduled Activity",
        "__UNNAMED__6": "Start_Action", "__UNNAMED__9": "End_Action"
    })
    .with_columns(pl.all().cast(pl.String, strict=False))
)

IEX = IEX.with_columns([
    pl.when(pl.col("Agent").str.contains("Generation Date: ", literal=True))
    .then(pl.col("Agent").str.extract(r"Generation Date: (.+)", 1))
    .otherwise(None).alias("Generate Date")
]).with_columns([
    pl.col("Generate Date").fill_null(strategy="backward"),
    pl.col("Agent").fill_null(strategy="forward")
])

offTable = IEX.filter(pl.col("Start_Shift") == "Off").with_columns([
    pl.col("Scheduled Activity").fill_null(pl.col("Start_Shift")),
    pl.lit(None, dtype=pl.String).alias("Start_Shift")
])

IEX_edit = (
    IEX.filter(
        (pl.col("Start_Shift").fill_null("") != "Off") & 
        (pl.col("Date").fill_null("") != "Date") & 
        ~(pl.col("Date").is_null() & pl.col("Scheduled Activity").is_null())
    )
    .with_columns([
        pl.col("Date").fill_null(strategy="forward"),
        pl.col("Start_Shift").fill_null(strategy="forward"),
        pl.col("End_Shift").fill_null(strategy="forward")
    ])
    .filter(pl.col("Scheduled Activity").is_not_null())
)

combined_df = pl.concat([offTable, IEX_edit], how="diagonal").with_columns([
    pl.col("Export time").cast(pl.String), 
    pl.col("Date").cast(pl.String),
    pl.col("Generate Date").cast(pl.String),
    pl.col("Date").str.slice(0, 4).cast(pl.Int32, strict=False).alias("Year"),
    pl.col("Date").alias("Month"),
    pl.col("Agent").str.extract(r"(\d+)", 1).cast(pl.Int64, strict=False).alias("IEX_ID")
])

max_generate_date_df = combined_df.group_by(["IEX_ID", "Date"]).agg(pl.col("Generate Date").max().alias("Max Generate Date"))

schedule_flags = ["Open Time", "Extra Hours", "No Call/No Show", "PTO", "Training Offline", "Sick Leave", "Paid Leave", "Termination", "Off Phone Misc", "Billable Training"]
productive_activities = ["Open Time", "Extra Hours"]
target_columns = ["Date", "OracleID", "IEX ID", "Agent Email", "First Shift", "Scheduled Activity", "Start_Action", "End_Action", "PST_Datetime", "Duration", "Work Category"]
compare_cols = ["PST_Date", "PST", "IST", "CLT", "VNT", "Agent Name", "First Shift", "LOB_Detail", "Connect State", "Scheduled Activity", "Start_Action", "End_Action", "Work Category"]

filtered_max_generate_date = (
    combined_df.join(max_generate_date_df, on=["IEX_ID", "Date"], how="left")
    .filter(pl.col("Generate Date") == pl.col("Max Generate Date"))
    .with_columns([
        pl.when(pl.col("Scheduled Activity").is_not_null().any().over(["Date", "IEX_ID"]))
        .then(1).otherwise(0).cast(pl.Int8, strict=False).alias("Scheduled"),
        pl.col("Export time").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S%.f", strict=False),
        pl.col("Date").str.strip_chars().str.strptime(pl.Date, "%m/%d/%y", strict=False),
        pl.col("Start_Shift").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("End_Shift").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("Start_Action").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("End_Action").str.strptime(pl.Time, "%I:%M %p", strict=False),
        pl.col("IEX_ID").cast(pl.Int64, strict=False)
    ])
    .with_columns([
        (pl.col("Start_Shift").dt.strftime("%H%M") + "-" + pl.col("End_Shift").dt.strftime("%H%M")).alias("Shift"),
        pl.when(pl.col("Start_Action").dt.minute() >= 30).then(pl.col("Start_Action").dt.strftime("%H") + ":30")
        .otherwise(pl.col("Start_Action").dt.strftime("%H") + ":00").alias("VNT")
    ])
    .with_columns([(pl.col("Date").cast(pl.String) + " " + pl.col("VNT")).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M", strict=False).dt.replace_time_zone("Asia/Ho_Chi_Minh").alias("VNT_Datetime")])
    .with_columns([pl.col("VNT_Datetime").dt.convert_time_zone("America/Los_Angeles").alias("PST_Datetime")])
    .with_columns([pl.col("VNT_Datetime").dt.replace_time_zone(None), pl.col("PST_Datetime").dt.replace_time_zone(None)])
)

intervals_df = (
    filtered_max_generate_date.with_columns([
        pl.col("Start_Action").fill_null(pl.col("Start_Shift")),
        pl.col("End_Action").fill_null(pl.col("End_Shift"))
    ])
    .filter(pl.col("Start_Action").is_not_null() & pl.col("End_Action").is_not_null())
    .with_columns([
        (pl.col("Date").cast(pl.String) + " " + pl.col("Start_Action").cast(pl.String)).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False).alias("Datetime_Start_Action"),
        (pl.col("Date").cast(pl.String) + " " + pl.col("End_Action").cast(pl.String)).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False).alias("Datetime_End_Action")
    ])
    .with_columns([
        pl.when(pl.col("Datetime_End_Action") < pl.col("Datetime_Start_Action")).then(pl.col("Datetime_End_Action") + pl.duration(days=1))
        .otherwise(pl.col("Datetime_End_Action")).alias("Datetime_End_Action")
    ])
    .with_columns([
        pl.col("Datetime_Start_Action").dt.truncate("30m").alias("Interval_Start_Floor"),
        pl.when(pl.col("Datetime_End_Action") == pl.col("Datetime_End_Action").dt.truncate("30m")).then(pl.col("Datetime_End_Action"))
        .otherwise(pl.col("Datetime_End_Action").dt.truncate("30m") + pl.duration(minutes=30)).alias("Interval_End_Ceil")
    ])
    .with_columns([pl.datetime_ranges(pl.col("Interval_Start_Floor"), pl.col("Interval_End_Ceil"), interval="30m", closed="left").alias("VNT_Intervals")])
    .explode("VNT_Intervals")
    .with_columns([
        (pl.col("VNT_Intervals") + pl.duration(minutes=30)).alias("Datetime_End_Time_Full"),
        pl.max_horizontal("VNT_Intervals", "Datetime_Start_Action").alias("Datetime_Start_Time")
    ])
    .with_columns([
        pl.min_horizontal("Datetime_End_Time_Full", "Datetime_End_Action").alias("Datetime_End_Time")
    ])
    .filter(pl.col("Datetime_Start_Time") < pl.col("Datetime_End_Time"))
    .with_columns([
        ((pl.col("Datetime_End_Time") - pl.col("Datetime_Start_Time")).dt.total_minutes() / 60.0).alias("Duration"),
        pl.col("VNT_Intervals").dt.replace_time_zone("Asia/Ho_Chi_Minh").dt.convert_time_zone("America/Los_Angeles").dt.replace_time_zone(None).alias("PST_Intervals"),
        pl.when(pl.col("Scheduled Activity").is_in(productive_activities)).then(pl.lit("Productive")).otherwise(pl.lit("Unproductive")).alias("Work Category")
    ])
    .with_columns([
        (pl.col("VNT_Intervals").dt.strftime("%H:%M") + "-" + pl.col("Datetime_End_Time_Full").dt.strftime("%H:%M")).alias("VNT_Interval_Range"),
        (pl.col("PST_Intervals").dt.strftime("%H:%M") + "-" + (pl.col("PST_Intervals") + pl.duration(minutes=30)).dt.strftime("%H:%M")).alias("PST_Interval_Range")
    ])
    .drop(["Interval_Start_Floor", "Interval_End_Ceil", "Datetime_End_Time_Full", "PST_Datetime"], strict=False)
)

intervals_df = (
    intervals_df.rename({"IEX_ID": "IEX ID", "Shift": "First Shift", "PST_Intervals": "PST_Datetime"})
    .select([c for c in target_columns if c in intervals_df.columns or c in ["Date", "IEX ID", "First Shift", "PST_Datetime"]])
    .with_columns([pl.col("Date").cast(pl.Date, strict=False), pl.col("IEX ID").cast(pl.Int64, strict=False)])
    .join(hc_extend_combination, on=["Date", "IEX ID"], how="left")
    .rename({"Email Id": "Agent Email"}, strict=False)
)

intervals_df = intervals_df.select([c for c in ["Date", "IEX ID", "Employee Name", "LOB", "Agent Email", "Supervisor Name"] if c in intervals_df.columns] + [c for c in intervals_df.columns if c not in ["Date", "IEX ID", "Employee Name", "LOB", "Agent Email", "Supervisor Name"]])

df_all_details = outage_db.filter(pl.col("PST_Datetime") == latest_dt).select(["Agent Name", "Agent Email", "LOB", "Business Location", "Connect State", "Duration", "PST_Datetime"]).pipe(add_time_cols)

df_compare = (
    df_all_details.filter(pl.col("Business Location") == "Concentrix (Ho Chi Minh City)")
    .join(
        intervals_df.with_columns([
            pl.col("PST_Datetime").cast(pl.Datetime, strict=False),
            pl.col("Agent Email").cast(pl.String).str.to_lowercase()
        ]),
        left_on=[pl.col("PST_Datetime").cast(pl.Datetime, strict=False), pl.col("Agent Email").cast(pl.String).str.to_lowercase()],
        right_on=["PST_Datetime", "Agent Email"], how="left"
    )
    .rename({"LOB_right": "LOB_Detail"}, strict=False)
)

display_order = ["PST_Date", "PST", "IST", "CLT", "VNT", "Agent Name", "Supervisor Name", "First Shift", "LOB_Detail", "Duration", "Connect State", "Scheduled Activity", "Start_Action", "End_Action", "Work Category"]
df_compare = df_compare.select([c for c in ["LOB"] + display_order if c in df_compare.columns])

df_lodging_active = df_compare.filter((pl.col("LOB") == "Lodging Chat") & pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)
df_lodging_inactive = df_compare.filter((pl.col("LOB") == "Lodging Chat") & ~pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)

df_non_lodging_active = df_compare.filter((pl.col("LOB") == "Non Lodging Chat") & pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)
df_non_lodging_inactive = df_compare.filter((pl.col("LOB") == "Non Lodging Chat") & ~pl.col("Connect State").is_in(ACTIVE_STATES)).sort(["Connect State"], nulls_last=True)

with pl.Config(tbl_rows=-1):
    display(df_lodging_active)
    display(df_lodging_inactive)
    display(df_non_lodging_active)
    display(df_non_lodging_inactive)


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Ngo, Tan Phat""","""Chau Thien Kim""","""0700-1600""","""Lodging""",00:13:44,"""AVAILABLECHAT""","""Open Time""",10:15:00,11:45:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Dinh Tuan""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:01:41,"""AVAILABLECHAT""","""Open Time""",11:00:00,13:15:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Thi Thu Thuy""","""Truong Thien Thanh Toan""","""0700-1600""","""Lodging""",00:00:50,"""AVAILABLECHAT""","""Open Time""",10:15:00,12:15:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Ta, Khanh Hoa""","""Chau Thien Kim""","""0700-1600""","""Lodging""",00:00:13,"""AVAILABLECHAT""","""Open Time""",10:15:00,11:45:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""To, Nguyen Hoang Long""","""Mia Minh Le""","""0700-1600""","""Lodging""",00:05:27,"""AVAILABLECHAT""","""Open Time""",08:00:00,12:15:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Vo, Duc Huy""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:00:40,"""AVAILABLECHAT""","""Open Time""",09:15:00,11:15:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Vo, Duc Huy""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:00:40,"""AVAILABLECHAT""","""Lunch""",11:15:00,12:15:00,"""Unproductive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Tan Nhat Truong""","""Ann""","""0700-1600""","""Lodging""",00:03:25,"""OUTBOUNDCALL""","""Open Time""",10:15:00,12:15:00,"""Productive"""


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Chau, Thien Kim""",null,null,null,03:40:22,"""ENDOFSHIFT""",null,null,null,null
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Tran, Hoang My Anh""",null,null,null,00:42:27,"""LOGIN""",null,null,null,null
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Tran, Thao Uyen""",null,null,null,00:34:15,"""LOGIN""",null,null,null,null
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Hoang, Dai Hai""","""Chau Thien Kim""","""0600-1500""","""Lodging""",00:01:53,"""LUNCH""","""Lunch""",11:00:00,12:00:00,"""Unproductive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Hoang, Le Vy""","""Ann""","""0600-1500""","""Lodging""",00:47:36,"""LUNCH""","""Lunch""",10:15:00,11:15:00,"""Unproductive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Hoang, Le Vy""","""Ann""","""0600-1500""","""Lodging""",00:47:36,"""LUNCH""","""Open Time""",11:15:00,13:00:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Le, Hoang Mai Thy""","""Tran Thao Uyen""","""0600-1500""","""Lodging""",00:52:23,"""LUNCH""","""Lunch""",10:15:00,11:15:00,"""Unproductive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Le, Hoang Mai Thy""","""Tran Thao Uyen""","""0600-1500""","""Lodging""",00:52:23,"""LUNCH""","""Open Time""",11:15:00,13:15:00,"""Productive"""
"""Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Le, Thuy Trieu""","""Ann""","""0600-1500""","""Lodging""",00:06:51,"""LUNCH""","""Lunch""",11:00:00,12:00:00,"""Unproductive"""


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Huy, Nguyen Nhat Anh""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:07:13,"""AVAILABLECHAT""","""Open Time""",11:00:00,13:45:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Lai, Ngoc Ai Linh""","""Mia Minh Le""","""0700-1600""","""Non_Lodging""",00:05:30,"""AVAILABLECHAT""","""Open Time""",08:30:00,11:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Quynh Tuyet Kha""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:15:17,"""AVAILABLECHAT""","""Open Time""",08:30:00,11:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Thanh Luan""","""Chau Thien Kim""","""0600-1500""","""Non_Lodging""",00:05:07,"""AVAILABLECHAT""","""Open Time""",07:30:00,11:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Vu, Minh Nghia""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:05:51,"""AVAILABLECHAT""","""Open Time""",10:45:00,12:30:00,"""Productive"""


LOB,PST_Date,PST,IST,CLT,VNT,Agent Name,Supervisor Name,First Shift,LOB_Detail,Duration,Connect State,Scheduled Activity,Start_Action,End_Action,Work Category
str,date,str,str,str,str,str,str,str,str,time,str,str,time,time,str
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Phung, Ha Tran""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:04:43,"""BREAK""","""Open Time""",11:00:00,12:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Phan, Thi Kim Tuyen""","""Tran Thao Uyen""","""0600-1500""","""Non_Lodging""",00:12:31,"""BREAK""","""Open Time""",10:45:00,12:30:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Nguyen, Dang Khoa""","""Tran Thao Uyen""","""0600-1500""","""Non_Lodging""",00:13:10,"""BREAK""","""Open Time""",11:00:00,13:00:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Le, Nguyen Dan Anh""","""Mia Minh Le""","""0600-1500""","""Non_Lodging""",00:03:23,"""BREAK""","""Open Time""",11:00:00,13:00:00,"""Productive"""
"""Non Lodging Chat""",2026-05-11,"""21:00""","""09:30""","""07:00""","""11:00""","""Hieu, Nguyen""",null,null,null,00:00:11,"""LOGIN""",null,null,null,null


In [7]:
TARGET_DATE = "2026-05-12"
START_TIME = "06:00"
END_TIME = "18:00"

df_summary = (
    intervals_df
    .with_columns([
        pl.col("PST_Datetime")
          .dt.replace_time_zone("America/Los_Angeles")
          .dt.convert_time_zone("Asia/Ho_Chi_Minh")
          .dt.replace_time_zone(None)
          .alias("VN_Datetime")
    ])
    .with_columns([
        pl.col("VN_Datetime").dt.date().alias("VN_Date"),
        pl.col("VN_Datetime").dt.strftime("%H:%M").alias("VN_Interval"),
        pl.col("PST_Datetime").dt.date().alias("PST_Date"),
        pl.col("PST_Datetime").dt.strftime("%H:%M").alias("PST_Interval"),

        pl.when(pl.col("Scheduled Activity").is_in(["Open Time", "Extra Hours"])).then(pl.lit("Available"))
          .when(pl.col("Scheduled Activity") == "Break").then(pl.lit("Break"))
          .when(pl.col("Scheduled Activity") == "Lunch").then(pl.lit("Lunch"))
          .when(pl.col("Scheduled Activity").str.contains("(?i)Training")).then(pl.lit("Training"))
          .when(pl.col("Scheduled Activity").is_in(["PTO", "Paid Leave"])).then(pl.lit("PL"))
          .when(pl.col("Scheduled Activity") == "No Call/No Show").then(pl.lit("NCNS"))
          .otherwise(pl.lit("Other"))
          .alias("Activity_Group"),

        (pl.col("Duration") * 2).alias("Headcount"),
    ])
    .pivot(
        values="Headcount",
        index=["VN_Date", "PST_Date", "VN_Interval", "PST_Interval", "LOB"],
        columns="Activity_Group",
        aggregate_function="sum"
    )
    .fill_null(0)
)

for col in ["Available", "Break", "Lunch", "Training", "PL", "NCNS"]:
    if col not in df_summary.columns:
        df_summary = df_summary.with_columns(pl.lit(0).alias(col))

custom_intervals = [
    i for i in [f"{h:02d}:{m:02d}" for h in range(24) for m in (0, 30)]
    if START_TIME <= i <= END_TIME
]

grid_df = (
    pl.DataFrame({
        "VN_Date": [TARGET_DATE] * 4,
        "LOB": ["Lodging", "Lodging_Nesting", "Non_Lodging", "Non_Lodging_Nesting"]
    })
    .with_columns(pl.col("VN_Date").str.strptime(pl.Date, "%Y-%m-%d"))
    .join(pl.DataFrame({"VN_Interval": custom_intervals}), how="cross")
    .with_columns([
        (pl.col("VN_Date").cast(pl.String) + " " + pl.col("VN_Interval"))
          .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M")
          .dt.replace_time_zone("Asia/Ho_Chi_Minh")
          .alias("VN_Datetime")
    ])
    .with_columns([
        pl.col("VN_Datetime").dt.convert_time_zone("America/Los_Angeles").alias("PST_Datetime")
    ])
    .with_columns([
        pl.col("PST_Datetime").dt.date().alias("PST_Date"),
        pl.col("PST_Datetime").dt.strftime("%H:%M").alias("PST_Interval")
    ])
    .select(["VN_Date", "PST_Date", "VN_Interval", "PST_Interval", "LOB"])
)

df_summary_full = (
    grid_df
    .join(
        df_summary.select(["VN_Date", "VN_Interval", "LOB",
                           "Available", "Break", "Lunch", "Training", "PL", "NCNS"]),
        on=["VN_Date", "VN_Interval", "LOB"],
        how="left"
    )
    .fill_null(0)
    .sort(["VN_Date", "LOB", "VN_Interval"])
)

num_cols  = ["Available", "Break", "Lunch", "Training", "PL", "NCNS"]
date_cols = ["VN_Date", "PST_Date"]
time_keys = ["VN_Date", "PST_Date", "VN_Interval", "PST_Interval"]

def aggregate_lob(df, lob_list):
    return (
        df.filter(pl.col("LOB").is_in(lob_list))
          .group_by(time_keys)
          .agg([pl.col(c).sum() for c in num_cols])
          .sort("VN_Interval")
    )

df_LG_final = aggregate_lob(df_summary_full, ["Lodging", "Lodging_Nesting"])
df_NL_final = aggregate_lob(df_summary_full, ["Non_Lodging", "Non_Lodging_Nesting"])

def format_date_col(series):
    """Convert date/datetime column sang string dd/mm trước khi style."""
    import pandas as pd
    if pd.api.types.is_datetime64_any_dtype(series):
        return series.dt.strftime('%d/%m')
    return series.apply(
        lambda x: x.strftime('%d/%m') if hasattr(x, 'strftime') else str(x)
    )

fmt_num  = {c: "{:.2f}" for c in num_cols}
fmt_date = {c: lambda x: x.strftime('%d/%m') if hasattr(x, 'strftime') else str(x)
            for c in date_cols}
fmt = {**fmt_num, **fmt_date}

def get_styled_html(df, title):
    pdf = df.to_pandas()

    for c in date_cols:
        if c in pdf.columns:
            pdf[c] = pdf[c].apply(
                lambda x: x.strftime('%d/%m') if hasattr(x, 'strftime') else str(x)
            )

    fmt_final = {c: "{:.2f}" for c in num_cols}

    return (
        pdf.style
           .hide(axis="index")
           .set_table_attributes('style="margin-right: 20px; text-align: center; border-collapse: collapse;"')
           .set_caption(f"<b style='font-size: 16px; color: #333;'>{title}</b>")
           .format(fmt_final)
           # ✅ Available/Break/Lunch/Training: nhiều = tốt → RdYlGn
           .background_gradient(cmap="RdYlGn", subset=["Available", "Break", "Lunch", "Training"])
           # ✅ PL/NCNS: nhiều = xấu → RdYlGn_r (đảo ngược)
           .background_gradient(cmap="RdYlGn_r", subset=["PL", "NCNS"])
           .to_html()
    )

html_output = f"""
<div style="display: flex; align-items: flex-start; overflow-x: auto; padding: 10px;">
    {get_styled_html(df_LG_final, "Lodging")}
    {get_styled_html(df_NL_final, "Non-Lodging")}
</div>
"""

display(HTML(html_output))

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_15308\537196953.py:31: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  .pivot(


VN_Date,PST_Date,VN_Interval,PST_Interval,Available,Break,Lunch,Training,PL,NCNS
12/05,11/05,06:00,16:00,19.00,0.00,0.00,7.00,1.00,0.00
12/05,11/05,06:30,16:30,19.00,0.00,0.00,7.00,1.00,0.00
12/05,11/05,07:00,17:00,19.50,1.50,0.00,7.00,2.00,0.00
12/05,11/05,07:30,17:30,17.00,4.00,0.00,7.00,2.00,0.00
12/05,11/05,08:00,18:00,23.00,5.00,0.00,0.00,2.00,0.00
12/05,11/05,08:30,18:30,28.00,0.00,0.00,0.00,2.00,0.00
12/05,11/05,09:00,19:00,19.50,1.50,7.00,0.00,2.00,0.00
12/05,11/05,09:30,19:30,21.00,0.00,7.00,0.00,2.00,0.00
12/05,11/05,10:00,20:00,15.00,2.00,12.50,0.00,2.00,0.00
12/05,11/05,10:30,20:30,15.00,0.00,15.50,0.00,1.50,0.00


In [8]:
SITE_FULLNAME = {"PUN": "Pune", "HCM": "Ho Chi Minh", "KOL": "Kolkata", "CAI": "Cairo"}

df_merge = (
    df_active.join(
        df_inactive.select(["LOB", "Site", "Heads Inactive"]),
        on=["LOB", "Site"],
        how="left"
    )
    .with_columns(pl.col("Site").replace(SITE_FULLNAME))
    .sort(["LOB", "Site"])
)

pdf = df_merge.to_pandas()
lodging_detail = df_lodging.to_pandas()
non_lodging_detail = df_non_lodging.to_pandas()

def build_summary_table(lob_name):

    sub = pdf[pdf["LOB"] == lob_name]

    html = f"""
    <div style="width:49%">
    <h3 style="text-align:center">{lob_name}</h3>

    <table border="1" style="width:100%;border-collapse:collapse;text-align:center;font-family:Arial;font-size:12px">
    <tr style="background:#222;color:white">
    <th>PST Date</th><th>PST</th><th>IST</th><th>CLT</th><th>VNT</th>
    <th>Site</th><th>Total OU Req</th><th>OU Req by Site</th>
    <th>Heads Active</th><th>Heads Inactive</th>
    <th>Missing Heads</th><th>Heads Active / OU Req by Site</th>
    </tr>
    """

    for i, (_, r) in enumerate(sub.iterrows()):

        html += "<tr>"

        if i == 0:

            rowspan = len(sub)

            html += f"""
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST_Date'].strftime('%Y-%m-%d')}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['IST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['CLT']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['VNT']}</td>
            """

        html += f"""
        <td><b>{r['Site']}</b></td>
        <td>{r['Total OU Req']}</td>
        <td>{r['OU Req by Site']}</td>
        <td>{r['Heads Active']}</td>
        <td>{r['Heads Inactive']}</td>
        <td style="
        background:{
            '#d32f2f' if r['Missing Heads'] < 0
            else '#2e7d32' if r['Missing Heads'] > 0
            else 'transparent'
        };
        color:white;
        font-weight:bold
        ">
        {r['Missing Heads']}
        </td>
        <td>{r['Heads Active / OU Req by Site']}</td>
        </tr>
        """

    html += "</table></div>"

    return html

def build_detail_table(df, title):

    html = f"""
    <div style="width:49%">
    <h3 style="text-align:center">{title} Details</h3>

    <table border="1" style="width:100%;border-collapse:collapse;text-align:center;font-family:Arial;font-size:12px">
    <tr style="background:#222;color:white">
    <th>PST Date</th><th>PST</th><th>IST</th><th>CLT</th><th>VNT</th>
    <th>Agent Name</th><th>Agent Email</th><th>Business Location</th><th>Connect State</th><th>Duration</th>
    </tr>
    """

    rowspan = len(df)

    for i, (_, r) in enumerate(df.iterrows()):

        html += "<tr>"

        if i == 0:

            html += f"""
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST_Date'].strftime('%Y-%m-%d')}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['IST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['CLT']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['VNT']}</td>
            """

        html += f"""
        <td>{r['Agent Name']}</td>
        <td>{r['Agent Email']}</td>
        <td>{r['Business Location']}</td>
        <td>{r['Connect State']}</td>
        <td>{r['Duration']}</td>
        </tr>
        """

    html += "</table></div>"

    return html

html = f"""
<style>
table {{
    border-collapse:collapse;
    text-align:center;
    font-family:Arial;
    font-size:13px;
    table-layout:auto;
}}

td, th {{
    padding:6px 10px;
    white-space:nowrap;
}}

th {{
    background:#222;
    color:white;
}}
</style>

<div style="margin-bottom:25px">
{build_summary_table("Lodging Chat")}
</div>

<div style="margin-bottom:40px">
{build_summary_table("Non Lodging Chat")}
</div>

<div style="margin-bottom:25px">
{build_detail_table(lodging_detail, "Lodging Chat")}
</div>

<div>
{build_detail_table(non_lodging_detail, "Non Lodging Chat")}
</div>
"""

display(HTML(html))